## Assignment 3 | 1905074


#### Imports

In [ ]:
from datasets import load_dataset
import numpy as np
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt
from sklearn.metrics import f1_score, confusion_matrix
import seaborn as sns
import pickle

#### Dataset Loading

In [ ]:
ds = load_dataset('fashion_mnist')
train_data = ds['train']
test_data = ds['test']

In [ ]:
def preprocess_data(data):
    X = np.stack([np.array(img, dtype=np.float32).reshape(28*28) / 255.0 for img in data['image']])
    y = np.eye(10)[data['label']]
    return X, y

# Preprocess datasets
X_train, y_train_onehot = preprocess_data(train_data)
X_test, y_test_onehot = preprocess_data(test_data)

# Split train data into training and validation sets
X_train, X_val, y_train_onehot, y_val_onehot = train_test_split(
    X_train, y_train_onehot, test_size=0.2, random_state=42)

print("Training data shape:", X_train.shape, y_train_onehot.shape)
print("Validation data shape:", X_val.shape, y_val_onehot.shape)
print("Test data shape:", X_test.shape, y_test_onehot.shape)

#### Neural Network

In [ ]:
class BatchNormalization:
    def __init__(self, input_dim, epsilon=1e-8):
        self.gamma = np.ones((1, input_dim))
        self.beta = np.zeros((1, input_dim))
        self.epsilon = epsilon

    def forward(self, x, training=True):
        if training:
            self.mean = np.mean(x, axis=0)
            self.variance = np.var(x, axis=0)
        self.x_norm = (x - self.mean) / np.sqrt(self.variance + self.epsilon)
        self.output = self.gamma * self.x_norm + self.beta
        return self.output

    def backward(self, d_output):
        m = d_output.shape[0]
        d_x_norm = d_output * self.gamma

        d_variance = np.sum(d_x_norm * (self.x_norm * -0.5 * np.power(self.variance + self.epsilon, -1.5)), axis=0)
        d_mean = np.sum(d_x_norm * -1 / np.sqrt(self.variance + self.epsilon), axis=0) + d_variance * np.mean(-2 * self.x_norm, axis=0)
        
        d_input = (d_x_norm / np.sqrt(self.variance + self.epsilon)) + (d_variance * 2 * self.x_norm / m) + (d_mean / m)

        self.d_gamma = np.sum(d_output * self.x_norm, axis=0)
        self.d_beta = np.sum(d_output, axis=0)

        return d_input


class ReLU:
    def forward(self, x):
        self.output = np.maximum(0, x)
        return self.output

    def backward(self, d_output):
        return d_output * (self.output > 0)


class Softmax:
    def forward(self, x):
        exp_values = np.exp(x - np.max(x, axis=1, keepdims=True))
        probabilities = exp_values / np.sum(exp_values, axis=1, keepdims=True)
        return probabilities


class Dropout:
    def __init__(self, dropout_rate):
        self.dropout_rate = dropout_rate

    def forward(self, x, training=True):
        if training:
            self.mask = np.random.binomial(1, 1 - self.dropout_rate, size=x.shape) / (1 - self.dropout_rate)
            return x * self.mask
        return x

    def backward(self, d_output):
        return d_output * self.mask

class AdamOptimizer:
    def __init__(self, learning_rate=0.01, beta1=0.9, beta2=0.999, epsilon=1e-8):
        self.learning_rate = learning_rate
        self.beta1 = beta1
        self.beta2 = beta2
        self.epsilon = epsilon
        self.t = 0  

    def initialize(self, weights_shape, biases_shape):
        # Initialize moments for weights and biases
        self.m_weights = np.zeros(weights_shape)
        self.v_weights = np.zeros(weights_shape)
        self.m_biases = np.zeros(biases_shape)
        self.v_biases = np.zeros(biases_shape)

    def update(self, weights, biases, d_weights, d_biases):
        self.t += 1

        self.m_weights = self.beta1 * self.m_weights + (1 - self.beta1) * d_weights
        self.v_weights = self.beta2 * self.v_weights + (1 - self.beta2) * (d_weights ** 2)

        m_hat_weights = self.m_weights / (1 - self.beta1 ** self.t)
        v_hat_weights = self.v_weights / (1 - self.beta2 ** self.t)
        
        weights -= self.learning_rate * m_hat_weights / (np.sqrt(v_hat_weights) + self.epsilon)

        self.m_biases = self.beta1 * self.m_biases + (1 - self.beta1) * d_biases
        self.v_biases = self.beta2 * self.v_biases + (1 - self.beta2) * (d_biases ** 2)

        m_hat_biases = self.m_biases / (1 - self.beta1 ** self.t)
        v_hat_biases = self.v_biases / (1 - self.beta2 ** self.t)

        biases -= self.learning_rate * m_hat_biases / (np.sqrt(v_hat_biases) + self.epsilon)

        return weights, biases

class DenseLayer:
    def __init__(self, input_size, output_size, learning_rate=0.01):
        self.weights = np.random.randn(input_size, output_size) * 0.01
        self.biases = np.zeros((1, output_size))
        self.optimizer = AdamOptimizer(learning_rate)
        self.optimizer.initialize(self.weights.shape, self.biases.shape)

    def forward(self, input_data):
        self.input_data = input_data
        self.output = np.dot(input_data, self.weights) + self.biases
        return self.output

    def backward(self, d_output):
        d_weights = np.dot(self.input_data.T, d_output)
        d_biases = np.sum(d_output, axis=0, keepdims=True)

        self.weights, self.biases = self.optimizer.update(self.weights, self.biases, d_weights, d_biases)

        input_error = np.dot(d_output, self.weights.T)

        return input_error

class NeuralNetwork:
    def __init__(self, layer_sizes, dropout_rate=0.5, learning_rate=0.01):
        layers = []

        for i in range(len(layer_sizes) - 1):
            layers.append(DenseLayer(layer_sizes[i], layer_sizes[i+1], learning_rate))
            if i < len(layer_sizes) - 2:  
                layers.append(BatchNormalization(layer_sizes[i+1]))
                layers.append(ReLU())
                layers.append(Dropout(dropout_rate))

        layers.append(Softmax())
        self.layers = layers

    def forward(self, x, training=True):
        for layer in self.layers:
            if isinstance(layer, (Dropout, BatchNormalization)):
                x = layer.forward(x, training)
            else:
                x = layer.forward(x)
        return x

    def backward(self, predictions, y):
        output_error = predictions - y
        for layer in reversed(self.layers[:-1]):
            output_error = layer.backward(output_error)

    def train(self, X_train, y_train, X_val, y_val, epochs=10, batch_size=32):
        history = {'loss': [], 'val_loss': [], 'accuracy': [], 'val_accuracy': [], 'val_f1': []}

        for epoch in range(epochs):
            for i in range(0, X_train.shape[0], batch_size):
                x_batch = X_train[i:i+batch_size]
                y_batch = y_train[i:i+batch_size]
                predictions = self.forward(x_batch, training=True)
                self.backward(predictions, y_batch)

            # Evaluate on training and validation data
            train_pred = self.forward(X_train, training=False)
            val_pred = self.forward(X_val, training=False)

            
            history['loss'].append(self.compute_loss(train_pred, y_train))
            history['val_loss'].append(self.compute_loss(val_pred, y_val))
            history['accuracy'].append(self.compute_accuracy(train_pred, y_train))
            history['val_accuracy'].append(self.compute_accuracy(val_pred, y_val))
            history['val_f1'].append(self.compute_f1(val_pred, y_val))

            
            print(f"Epoch {epoch+1}/{epochs} - "
                  f"Loss: {history['loss'][-1]:.4f}, Accuracy: {history['accuracy'][-1]:.4f}, "
                  f"Val Loss: {history['val_loss'][-1]:.4f}, Val Accuracy: {history['val_accuracy'][-1]:.4f}, "
                  f"Val F1: {history['val_f1'][-1]:.4f}")

        return history

    def compute_loss(self, predictions, y):
        return -np.mean(y * np.log(predictions + 1e-8))

    def compute_accuracy(self, predictions, y):
        pred_classes = np.argmax(predictions, axis=1)
        true_classes = np.argmax(y, axis=1)
        return np.mean(pred_classes == true_classes)

    def compute_f1(self, predictions, y):
        pred_classes = np.argmax(predictions, axis=1)
        true_classes = np.argmax(y, axis=1)
        return f1_score(true_classes, pred_classes, average='macro')

    def plot_metrics(self, history):
        epochs = range(len(history['loss']))
        
        plt.figure(figsize=(12, 6))
        plt.subplot(1, 2, 1)
        plt.plot(epochs, history['loss'], label='Training Loss')
        plt.plot(epochs, history['val_loss'], label='Validation Loss')
        plt.title('Loss over Epochs')
        plt.xlabel('Epochs')
        plt.ylabel('Loss')
        plt.legend()

        plt.subplot(1, 2, 2)
        plt.plot(epochs, history['accuracy'], label='Training Accuracy')
        plt.plot(epochs, history['val_accuracy'], label='Validation Accuracy')
        plt.title('Accuracy over Epochs')
        plt.xlabel('Epochs')
        plt.ylabel('Accuracy')
        plt.legend()
        plt.show()

    def plot_confusion_matrix(self, X, y, title='Confusion Matrix'):
        pred_classes = np.argmax(self.forward(X, training=False), axis=1)
        true_classes = np.argmax(y, axis=1)
        cm = confusion_matrix(true_classes, pred_classes)

        plt.figure(figsize=(8, 6))
        sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
        plt.title(title)
        plt.xlabel('Predicted Label')
        plt.ylabel('True Label')
        plt.show()

    def save(self, filename):
        with open(filename, 'wb') as file:
            pickle.dump(self, file)
    
    @staticmethod
    def load(filename):
        with open(filename, 'rb') as file:
            model = pickle.load(file)
        return model

#### Train Model

In [ ]:
# layer_sizes = [784,128,64,32,16, 10]  

# nn = NeuralNetwork(layer_sizes,learning_rate=0.005)
# training_history=nn.train(X_train, y_train_onehot,X_val,y_val_onehot, epochs=10)
# nn.plot_metrics(training_history)

#### Load Model

In [ ]:
nn = NeuralNetwork.load('model_1905074.pickle')

#### Testing Model

In [ ]:
test_predictions = nn.forward(X_test, training=False)
pred_classes = np.argmax(test_predictions, axis=1)
# print(pred_classes)
test_loss = nn.compute_loss(test_predictions, y_test_onehot)
test_accuracy = nn.compute_accuracy(test_predictions, y_test_onehot)

print(f'Test Loss: {test_loss:.4f}, Test Accuracy: {test_accuracy:.4f}')

#### Report

In [ ]:
def evaluate_and_report(model, X_val, y_val):
    val_predictions = model.forward(X_val, training=False)
    
    val_loss = model.compute_loss(val_predictions, y_val)
    val_accuracy = model.compute_accuracy(val_predictions, y_val)
    val_f1 = f1_score(np.argmax(y_val, axis=1), np.argmax(val_predictions, axis=1), average='macro')

    print(f"Validation Loss: {val_loss:.4f}, Validation Accuracy: {val_accuracy:.4f}, Validation F1: {val_f1:.4f}")
    return val_loss, val_accuracy, val_f1


best_f1 = 0
best_model_params = {}

architectures = [
    [784, 128, 10],   
    [784, 64, 10],    
    [784, 128, 64, 10]  
]


for architecture in architectures:
    for lr in [0.005, 0.001, 0.0005]:
        print(f"\nTraining model with architecture {architecture} and learning rate {lr}")
        
        nn = NeuralNetwork(layer_sizes=architecture, learning_rate=lr)
        
        
        training_history = nn.train(X_train, y_train_onehot, X_val, y_val_onehot, epochs=10)

        
        val_loss, val_accuracy, val_f1 = evaluate_and_report(nn, X_val, y_val_onehot)

        
        nn.plot_metrics(training_history)

        nn.plot_confusion_matrix(X_test,y_test_onehot)

        # Update best model if needed
        if val_f1 > best_f1:
            best_f1 = val_f1
            best_model_params = {'architecture': architecture, 'learning_rate': lr, 'model': nn}


print(f"\nBest Model Architecture: {best_model_params['architecture']}, Learning Rate: {best_model_params['learning_rate']}")
best_model = best_model_params['model']
test_predictions = best_model.forward(X_test, training=False)
test_loss = best_model.compute_loss(test_predictions, y_test_onehot)
test_accuracy = best_model.compute_accuracy(test_predictions, y_test_onehot)
test_f1 = f1_score(np.argmax(y_test_onehot, axis=1), np.argmax(test_predictions, axis=1), average='macro')

print(f"Test Loss: {test_loss:.4f}, Test Accuracy: {test_accuracy:.4f}, Test F1: {test_f1:.4f}")


best_model.plot_confusion_matrix(X_test, y_test_onehot, title='Confusion Matrix on Test Set')